# Universal Stock Direction Prediction Using Genetic Algorithm and XGBoost with LSTM
*Alternative title: Universal Stock Price Direction Prediction Using GA-XGBoost and LSTM*

**Course:** Deep Learning mini-project  |  **Task:** next-day UP/DOWN classification of Indian (NSE) stocks  |  **Data:** Kaggle `manavanghan/nse-india-stock-market-data-2015-2024`

---
## How to run (Kaggle)
1. Upload this notebook to Kaggle (*Create → New Notebook → File → Import Notebook*).
2. **Settings → Internet: ON** (KaggleHub downloads the dataset). **Accelerator: GPU recommended** (only the LSTM benefits).
3. Change **only** `TARGET_STOCK` in the *Configuration* cell (e.g. `"TCS"`, `"RELIANCE"`, `"INFY"`, `"HDFCBANK"`, `"ICICIBANK"`, `"SBIN"`).
4. *Run All*.  Tip: set `QUICK_MODE = True` first for a fast smoke test, then switch to `False` for the final run.

## What the notebook builds
| # | Component | Notes |
|---|-----------|-------|
| 1 | Data cleaning (per `Symbol`+`Date`) | duplicates removed **before** any indicator is computed |
| 2 | ~80 technical indicators per stock | computed independently inside each stock, relative/stationary forms |
| 3 | Per-stock chronological split 70/15/15 | concatenated into universal train / validation / test sets |
| 4 | Exp 1-5 | OHLCV XGBoost, expanded-feature XGBoost, **Genetic-Algorithm feature selection + XGBoost**, universal LSTM, majority baseline |
| 5 | Evaluation | accuracy, precision, recall, F1, ROC-AUC, balanced accuracy, specificity, confusion matrix, per-stock table |
| 6 | Leakage audit | numerical + procedural checks, PASS/FAIL |
| 7 | Saved artifacts | everything a future Streamlit app needs |

## Honesty rules used throughout
* **No metric is hard-coded or "adjusted".** Every number is computed from the untouched test split.
* The test split is opened **once**, after every model has been trained, tuned and frozen (see the `open_test` guard and the audit).
* A ">90 %" accuracy is only an *experimental target*. Next-day direction of liquid stocks is close to a coin flip; results around 50-58 % are normal in the literature. If the notebook ever reports something far above that, treat it as a *red flag for leakage*, not a success.
* This is an **adaptation** of the GA-XGBoost methodology to NSE data. It does **not** reproduce the paper's KOSPI experiment and does not inherit the paper's reported accuracy.

In [ ]:
# ============================================================
# 1. SETUP: imports, optional installs, hardware check
# ============================================================
import os, sys, json, re, time, math, random, warnings, subprocess, difflib, shutil, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

def ensure_package(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure_package("kagglehub")
ensure_package("xgboost")
ensure_package("joblib")

import kagglehub, joblib
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, balanced_accuracy_score, confusion_matrix,
                             roc_curve, precision_recall_curve, average_precision_score,
                             classification_report)
import tensorflow as tf

try:
    import talib            # optional: only used for Hilbert-Transform cycle features
    HAS_TALIB = True
except Exception:
    HAS_TALIB = False

try:
    from IPython.display import display
except Exception:
    display = print

GPUS = tf.config.list_physical_devices("GPU")
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__, "| pandas:", pd.__version__)
print("XGBoost     :", xgb.__version__, "| TensorFlow:", tf.__version__)
print("GPU devices :", [g.name for g in GPUS] if GPUS else "none (LSTM will run on CPU - slower)")
print("TA-Lib      :", "available (Hilbert-Transform features enabled)" if HAS_TALIB
      else "not installed -> all core indicators use the built-in pandas/numpy implementation; HT features skipped")

In [ ]:
# ============================================================
# 2. CONFIGURATION  (the ONLY value you normally change is TARGET_STOCK)
# ============================================================
TARGET_STOCK = "TCS"          # <-- change this: "RELIANCE", "INFY", "HDFCBANK", "ICICIBANK", "SBIN", ...
QUICK_MODE   = False          # True = fast smoke test (small GA, few epochs); False = full paper-style run

# ---------------- advanced settings (defaults are fine) ----------------
SEED             = 42
DATASET_HANDLE   = "manavanghan/nse-india-stock-market-data-2015-2024"
USE_ADJ_CLOSE    = False      # False = follow the spec literally (raw Close). True = rescale OHLC by AdjClose/Close
MIN_ROWS_PER_STOCK = 1000     # stocks with less history are not eligible for the universal training universe
HARD_MIN_ROWS    = 400        # absolute minimum for priority stocks / target stock
PRIORITY_STOCKS  = ["TCS", "RELIANCE", "INFY", "HDFCBANK", "ICICIBANK", "SBIN"]   # always included if present
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15      # remaining 15 % = untouched test (per stock)
EMBARGO_ROWS     = 1          # rows purged at the end of train/val so that no label peeks into the next segment
WINDOW           = 60         # LSTM sequence length (trading days)
CLIP_Z           = 5.0        # LSTM inputs are standardised then clipped to +/- CLIP_Z
THRESHOLD_MODE   = "val_balanced_accuracy"   # decision threshold chosen on VALIDATION only ("fixed_0.5" to disable)
RUN_TUNING       = True       # optional separate XGBoost hyper-parameter search (train/validation only)
RUN_LOSO         = False      # optional leave-one-stock-out experiment (Mode 2)
LOSO_STOCKS      = None       # e.g. ["TCS", "RELIANCE"]; None -> target + a few priority stocks
RUN_SHAP         = False      # optional SHAP summary on a VALIDATION sample

if QUICK_MODE:
    MAX_STOCKS, GA_POP, GA_GENS, GA_TREES = 40, 6, 8, 20
    GA_TRAIN_ROWS, GA_VAL_ROWS = 40_000, 20_000
    LSTM_EPOCHS, TUNE_ITERS = 8, 6
else:
    MAX_STOCKS, GA_POP, GA_GENS, GA_TREES = 100, 8, 30, 50
    GA_TRAIN_ROWS, GA_VAL_ROWS = 150_000, 60_000
    LSTM_EPOCHS, TUNE_ITERS = 40, 20

# Genetic-algorithm parameters (inspired by the paper)
GA_CROSSOVER, GA_MUTATION, GA_ELITISM = 0.50, 0.375, 1
GA_TOURNAMENT_K   = 3
GA_MIN_FEATURES   = 5
GA_GENE_FLIP_COUNT = 3        # when a chromosome mutates, ~3 genes flip on average

# Transparent paper-style XGBoost baseline (n_estimators fixed at 100)
XGB_PARAMS = dict(n_estimators=100, max_depth=4, learning_rate=0.1, subsample=0.8,
                  colsample_bytree=0.8, objective="binary:logistic", eval_metric="logloss",
                  tree_method="hist", n_jobs=-1, random_state=SEED)
LSTM_BATCH, LSTM_PATIENCE = 512, 5

# ---------------- output folder ----------------
WORK_DIR    = Path("/kaggle/working") if os.path.isdir("/kaggle/working") else Path.cwd()
PROJECT_DIR = WORK_DIR / "universal_stock_project"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- reproducibility + audit bookkeeping ----------------
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

EVENTS = []   # chronological log used by the leakage audit
USED   = {}   # which split labels each stage actually touched (verified in the audit)
def log_event(name, **info):
    EVENTS.append({"order": len(EVENTS), "event": name, "time": time.strftime("%H:%M:%S"), **info})
def register_use(stage, split_array, idx):
    USED.setdefault(stage, set()).update(np.unique(split_array[idx]).tolist())

print(f"TARGET_STOCK = {TARGET_STOCK!r} | QUICK_MODE = {QUICK_MODE}")
print(f"Universe size <= {MAX_STOCKS} | GA: pop={GA_POP}, gens={GA_GENS}, crossover={GA_CROSSOVER}, mutation={GA_MUTATION}, elitism={GA_ELITISM}")
print(f"Outputs will be written to: {PROJECT_DIR}")

## 3. Dataset download (KaggleHub) and file discovery
The dataset is downloaded directly with `kagglehub.dataset_download(...)`; no manual CSV upload is needed. If the download fails, the exact error is printed and `/kaggle/input/` is searched - but **only** the file `nifty500_stocks.csv` (the intended NSE dataset) is accepted as a fallback. Nothing else is silently substituted.

In [ ]:
DATA_FILE = None
dataset_path = None
try:
    dataset_path = kagglehub.dataset_download(DATASET_HANDLE)
    print("KaggleHub download OK ->", dataset_path)
except Exception as e:
    print("=" * 70)
    print("KaggleHub download FAILED. Exact error:")
    print(f"{type(e).__name__}: {e}")
    traceback.print_exc()
    print("=" * 70)
    print("Checking /kaggle/input/ for the intended dataset ...")
    if os.path.isdir("/kaggle/input"):
        for p in sorted(Path("/kaggle/input").iterdir()):
            print("   ", p)
        hits = sorted(Path("/kaggle/input").rglob("nifty500_stocks.csv"))
        if hits:
            dataset_path = str(hits[0].parent)
            print(f"\nFound the intended NSE file nifty500_stocks.csv in {dataset_path}. Using it.")
        else:
            print("\nnifty500_stocks.csv was NOT found in /kaggle/input/.")
    else:
        print("/kaggle/input does not exist (not running on Kaggle?).")
    if dataset_path is None:
        raise RuntimeError("Could not obtain the NSE dataset. Turn Internet ON in the Kaggle notebook settings "
                           "(or attach the dataset manually) and run again. No other dataset was substituted.")

csv_files = sorted(Path(dataset_path).rglob("*.csv"))
print(f"\nCSV files found under {dataset_path}:")
for p in csv_files:
    print(f"  {p.name:45s} {p.stat().st_size/1e6:9.2f} MB")
if not csv_files:
    raise RuntimeError("The downloaded dataset folder contains no CSV files.")

main = [p for p in csv_files if p.name.lower() == "nifty500_stocks.csv"]
if main:
    DATA_FILE = main[0]
else:
    print("\nWARNING: expected file nifty500_stocks.csv not found; scanning headers for a stock-price table ...")
    for p in sorted(csv_files, key=lambda q: -q.stat().st_size):
        head = pd.read_csv(p, nrows=5)
        lc = [c.lower().strip() for c in head.columns]
        if any(c in lc for c in ("symbol", "ticker", "stock", "company")) and "close" in lc:
            DATA_FILE = p
            break
    if DATA_FILE is None:
        raise RuntimeError("No CSV with a ticker column and a Close column was found in the dataset.")
print("\nUsing data file:", DATA_FILE)

## 4. Load and clean the data
Cleaning order (as required): parse dates → numeric OHLCV → drop missing → drop invalid/non-positive prices → **remove duplicate `Symbol`+`Date` rows** → sort by `Symbol`,`Date`.  All of this happens **before** any indicator is calculated.

In [ ]:
raw = pd.read_csv(DATA_FILE)
print("Raw shape  :", raw.shape)
print("Raw columns:", list(raw.columns))
print("\nFirst rows:"); display(raw.head())
print("\nMissing values per column (raw file):"); print(raw.isna().sum().to_string())

def find_col(columns, candidates):
    lookup = {re.sub(r"[\s_]+", " ", c.lower().strip()): c for c in columns}
    for cand in candidates:
        if cand in lookup:
            return lookup[cand]
    return None

col_map = {
    "Symbol": find_col(raw.columns, ["symbol", "ticker", "stock", "company"]),
    "Date":   find_col(raw.columns, ["date", "datetime", "timestamp", "time"]),
    "Open":   find_col(raw.columns, ["open"]),
    "High":   find_col(raw.columns, ["high"]),
    "Low":    find_col(raw.columns, ["low"]),
    "Close":  find_col(raw.columns, ["close"]),
    "Volume": find_col(raw.columns, ["volume", "vol"]),
    "AdjClose": find_col(raw.columns, ["adj close", "adjclose", "adjusted close"]),
}
print("\nDetected column mapping:", col_map)
missing_required = [k for k in ["Symbol", "Date", "Open", "High", "Low", "Close", "Volume"] if col_map[k] is None]
if missing_required:
    raise RuntimeError(f"Required columns not found: {missing_required}. Actual columns: {list(raw.columns)}")

df = pd.DataFrame({k: raw[v] for k, v in col_map.items() if v is not None})
del raw

def parse_dates(s):
    out = pd.to_datetime(s, errors="coerce")
    if out.dtype == object:                                   # mixed time-zone offsets
        out = pd.to_datetime(s, errors="coerce", utc=True).dt.tz_localize(None)
    elif getattr(out.dt, "tz", None) is not None:
        out = out.dt.tz_localize(None)
    return out

n_raw = len(df)
df["Date"] = parse_dates(df["Date"])
df["Symbol"] = df["Symbol"].astype(str).str.strip()
num_cols = ["Open", "High", "Low", "Close", "Volume"] + (["AdjClose"] if "AdjClose" in df else [])
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

required = ["Symbol", "Date", "Open", "High", "Low", "Close", "Volume"]
print("\nMissing required values after type conversion:"); print(df[required].isna().sum().to_string())

df = df.dropna(subset=required)
n_after_na = len(df)
df = df[(df[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (df["Volume"] >= 0)]
df = df[df["High"] >= df["Low"]]
n_after_invalid = len(df)
dup_count = int(df.duplicated(subset=["Symbol", "Date"]).sum())
df = df.drop_duplicates(subset=["Symbol", "Date"], keep="last")
df = df.sort_values(["Symbol", "Date"]).reset_index(drop=True)

if USE_ADJ_CLOSE and "AdjClose" in df:
    factor = (df["AdjClose"] / df["Close"]).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    for c in ["Open", "High", "Low", "Close"]:
        df[c] = df[c] * factor
    print("USE_ADJ_CLOSE=True -> OHLC rescaled by AdjClose/Close")

print("\n" + "=" * 60)
print("DATA CLEANING SUMMARY")
print("=" * 60)
print(f"Rows in raw file              : {n_raw:,}")
print(f"Rows after dropping missing   : {n_after_na:,}")
print(f"Rows after invalid-price drop : {n_after_invalid:,}")
print(f"Duplicate Symbol+Date removed : {dup_count:,}")
print(f"Total rows (clean)            : {len(df):,}")
print(f"Total stocks                  : {df['Symbol'].nunique():,}")
print(f"Date range                    : {df['Date'].min().date()}  ->  {df['Date'].max().date()}")
print(f"Min date / Max date           : {df['Date'].min().date()} / {df['Date'].max().date()}")
rows_per_stock = df.groupby("Symbol").size()
print("\nRows per stock:"); print(rows_per_stock.describe().round(1).to_string())
print("\nStocks with the most / fewest rows:")
print(rows_per_stock.sort_values(ascending=False).head(5).to_string()); print("..."); print(rows_per_stock.sort_values().head(5).to_string())
print("\nMissing values after cleaning:"); print(df.isna().sum().to_string())
print("Duplicate (Symbol, Date) pairs after cleaning:", int(df.duplicated(subset=["Symbol", "Date"]).sum()))

_r = df.groupby("Symbol")["Close"].pct_change()
n_extreme = int((_r.abs() > 0.40).sum())
print(f"\nDiagnostic: {n_extreme} daily moves larger than +/-40 % (often unadjusted splits/bonus issues).")
if n_extreme > 0 and not USE_ADJ_CLOSE:
    print("  -> They are kept because the spec defines the target on raw Close; set USE_ADJ_CLOSE=True to adjust.")
del _r

## 5. Resolve `TARGET_STOCK` and choose the universal training universe
* The ticker column is detected automatically; tickers are matched case-insensitively and a suffix such as `.NS` / `-EQ` is ignored.
* If `TARGET_STOCK` is not found, all valid tickers and the closest matches are shown and the notebook stops cleanly.
* **Universe selection** (never uses test-period information): stocks with ≥ `MIN_ROWS_PER_STOCK` rows are ranked by average traded value (Close × Volume) measured **only on the first 70 % of each stock's history**; the top `MAX_STOCKS` are kept. The priority stocks and the target stock are always included.
* The trained model does not depend on `TARGET_STOCK` unless the target is outside the top-liquidity universe (then it is added to the universe).

In [ ]:
def norm_symbol(s):
    return re.sub(r"(\.NS|\.BO|\.NSE|-EQ|\.EQ)$", "", str(s).upper().strip())

all_symbols = sorted(df["Symbol"].unique())
sym_map = {}
for s in all_symbols:
    sym_map.setdefault(norm_symbol(s), s)

key = norm_symbol(TARGET_STOCK)
if key not in sym_map:
    print(f"TARGET_STOCK {TARGET_STOCK!r} was not found in the dataset.\n")
    print(f"All {len(all_symbols)} valid tickers:")
    print(", ".join(all_symbols))
    close = difflib.get_close_matches(key, list(sym_map.keys()), n=8, cutoff=0.4)
    print("\nClosest matches:", close if close else "none")
    raise RuntimeError(f"TARGET_STOCK {TARGET_STOCK!r} not found. Pick one of the tickers printed above.")
TARGET_SYMBOL = sym_map[key]
print(f"TARGET_STOCK {TARGET_STOCK!r} resolved to dataset symbol {TARGET_SYMBOL!r}")

df["_pos"] = df.groupby("Symbol").cumcount()
df["_n"] = df.groupby("Symbol")["Close"].transform("size")
early = df[df["_pos"] < 0.70 * df["_n"]]
liquidity = (early["Close"] * early["Volume"]).groupby(early["Symbol"]).mean()
stats = pd.DataFrame({"n_rows": rows_per_stock,
                      "first_date": df.groupby("Symbol")["Date"].min(),
                      "last_date": df.groupby("Symbol")["Date"].max(),
                      "liquidity": liquidity}).fillna({"liquidity": 0.0})
df = df.drop(columns=["_pos", "_n"])

eligible = stats[stats["n_rows"] >= MIN_ROWS_PER_STOCK].sort_values("liquidity", ascending=False)
forced = []
for name in PRIORITY_STOCKS + [TARGET_STOCK]:
    k = norm_symbol(name)
    if k in sym_map and stats.loc[sym_map[k], "n_rows"] >= HARD_MIN_ROWS:
        forced.append(sym_map[k])
forced = list(dict.fromkeys(forced))
if TARGET_SYMBOL not in forced:
    raise RuntimeError(f"{TARGET_SYMBOL} has only {stats.loc[TARGET_SYMBOL, 'n_rows']} rows; need at least {HARD_MIN_ROWS}.")
rest = [s for s in eligible.index if s not in forced]
limit = MAX_STOCKS if MAX_STOCKS else len(eligible)
UNIVERSE = forced + rest[: max(0, limit - len(forced))]

df_u = df[df["Symbol"].isin(UNIVERSE)].sort_values(["Symbol", "Date"]).reset_index(drop=True)
print(f"\nEligible stocks (>= {MIN_ROWS_PER_STOCK} rows): {len(eligible)} of {len(all_symbols)}")
print(f"Universal training universe: {len(UNIVERSE)} stocks, {len(df_u):,} rows")
print("Forced/priority stocks included:", forced)
print("Universe preview:", UNIVERSE[:30], "..." if len(UNIVERSE) > 30 else "")

## 6. Feature engineering (technical-indicator expansion)
Implementation notes
* Every indicator is computed **independently inside each stock** (no rolling window ever crosses from one company to the next).
* Indicators are written in pandas/numpy using the standard TA-Lib formulas (Wilder smoothing for RSI/ATR/ADX etc.), so the notebook runs identically with or without TA-Lib. If TA-Lib is installed, Hilbert-Transform features are added as well (and fail gracefully otherwise).
* Almost everything is expressed in **relative / scale-free form** (returns, `Close/SMA - 1`, ATR/Close, volume ratios, …) so one model can be shared by stocks with very different price levels. No ticker one-hot is used, which is what allows the model to be applied to any stock.
* Only past data is used: no negative shifts, no centred windows. The feature code is stored as a string, executed, and **also written to `feature_engineering.py`** so the future Streamlit app computes exactly the same features.
* An optional NIFTY index/market feature set is intentionally not used: the primary model must work from a single stock's own history, which keeps inference simple.

In [ ]:
FEATURE_CODE = r'''
import numpy as np
import pandas as pd

try:
    import talib as _talib
except Exception:
    _talib = None

FEATURE_ENGINEERING_VERSION = "1.0"

def _sma(s, n):
    return s.rolling(n, min_periods=n).mean()

def _ema(s, n):
    return s.ewm(span=n, adjust=False, min_periods=n).mean()

def _wilder(s, n):
    return s.ewm(alpha=1.0 / n, adjust=False, min_periods=n).mean()

def _days_since_extreme(a, w, use_max):
    # number of bars since the max/min inside the trailing window of length w (past data only)
    out = np.full(len(a), np.nan)
    if len(a) >= w:
        win = np.lib.stride_tricks.sliding_window_view(a, w)[:, ::-1]
        out[w - 1:] = win.argmax(axis=1) if use_max else win.argmin(axis=1)
    return out

def build_features_for_stock(d):
    # d: OHLCV rows of ONE stock, sorted by Date ascending. Returns a float32 feature frame with the same index.
    assert d["Date"].is_monotonic_increasing, "rows must be sorted by Date inside each stock"
    o = d["Open"].astype(float); h = d["High"].astype(float); l = d["Low"].astype(float)
    c = d["Close"].astype(float); v = d["Volume"].astype(float)
    idx = d.index
    f = {}
    prev_c = c.shift(1)
    ret1 = c.pct_change(1)
    delta = c.diff()

    # ---- price / return features ----
    for k in (1, 2, 3, 5, 10, 20):
        f["ret_%dd" % k] = c.pct_change(k)
    f["hl_pct"] = (h - l) / c
    f["oc_pct"] = (c - o) / o
    f["gap_pct"] = o / prev_c - 1.0
    for k in (5, 10, 20):
        f["roll_vol_%d" % k] = ret1.rolling(k, min_periods=k).std()

    # ---- lag features (relative) ----
    for k in (1, 2, 3, 5, 10, 20):
        f["close_lag_%d_rel" % k] = c.shift(k) / c - 1.0
        f["ret_lag_%d" % k] = ret1.shift(k)

    # ---- moving averages: distance of price + slope ----
    for n in (5, 10, 20, 50, 100):
        s_ = _sma(c, n)
        e_ = _ema(c, n)
        f["sma_%d_dist" % n] = c / s_ - 1.0
        f["ema_%d_dist" % n] = c / e_ - 1.0
        if n in (10, 20, 50):
            f["sma_%d_slope" % n] = s_ / s_.shift(5) - 1.0
            f["ema_%d_slope" % n] = e_ / e_.shift(5) - 1.0

    # ---- momentum indicators ----
    gain = delta.clip(lower=0.0)
    loss = (-delta).clip(lower=0.0)
    rs = _wilder(gain, 14) / _wilder(loss, 14)
    f["rsi_14"] = 100.0 - 100.0 / (1.0 + rs)
    f["roc_12"] = (c / c.shift(12) - 1.0) * 100.0

    tp = (h + l + c) / 3.0
    tp_arr = tp.values
    mad = np.full(len(tp_arr), np.nan)
    if len(tp_arr) >= 20:
        win = np.lib.stride_tricks.sliding_window_view(tp_arr, 20)
        mad[19:] = np.abs(win - win.mean(axis=1, keepdims=True)).mean(axis=1)
    f["cci_20"] = (tp - _sma(tp, 20)) / (0.015 * pd.Series(mad, index=idx))

    hh14 = h.rolling(14, min_periods=14).max()
    ll14 = l.rolling(14, min_periods=14).min()
    rng14 = hh14 - ll14
    f["willr_14"] = -100.0 * (hh14 - c) / rng14
    stoch_k = 100.0 * (c - ll14) / rng14
    f["stoch_k_14"] = stoch_k
    f["stoch_d_3"] = _sma(stoch_k, 3)

    rmf = tp * v
    pos_mf = rmf.where(tp > tp.shift(1), 0.0)
    neg_mf = rmf.where(tp < tp.shift(1), 0.0)
    mfr = pos_mf.rolling(14, min_periods=14).sum() / neg_mf.rolling(14, min_periods=14).sum()
    f["mfi_14"] = 100.0 - 100.0 / (1.0 + mfr)

    tr = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
    up_move = h.diff()
    down_move = -l.diff()
    plus_dm = pd.Series(np.where((up_move > down_move) & (up_move > 0), up_move, 0.0), index=idx)
    minus_dm = pd.Series(np.where((down_move > up_move) & (down_move > 0), down_move, 0.0), index=idx)
    atr14 = _wilder(tr, 14)
    pdi = 100.0 * _wilder(plus_dm, 14) / atr14
    mdi = 100.0 * _wilder(minus_dm, 14) / atr14
    dx = 100.0 * (pdi - mdi).abs() / (pdi + mdi)
    f["adx_14"] = _wilder(dx, 14)
    f["plus_di_14"] = pdi
    f["minus_di_14"] = mdi
    f["dx_14"] = dx
    f["plus_dm_14"] = _wilder(plus_dm, 14) / c
    f["minus_dm_14"] = _wilder(minus_dm, 14) / c

    min_lc = pd.concat([l, prev_c], axis=1).min(axis=1)
    max_hc = pd.concat([h, prev_c], axis=1).max(axis=1)
    bp = c - min_lc
    trr = max_hc - min_lc
    def _avg(n):
        return bp.rolling(n, min_periods=n).sum() / trr.rolling(n, min_periods=n).sum()
    f["ultosc"] = 100.0 * (4.0 * _avg(7) + 2.0 * _avg(14) + _avg(28)) / 7.0

    t3 = _ema(_ema(_ema(c, 15), 15), 15)
    f["trix_15"] = t3.pct_change(1) * 100.0

    n_ar = 25
    ds_h = _days_since_extreme(h.values, n_ar + 1, True)
    ds_l = _days_since_extreme(l.values, n_ar + 1, False)
    aroon_up = pd.Series(100.0 * (n_ar - ds_h) / n_ar, index=idx)
    aroon_dn = pd.Series(100.0 * (n_ar - ds_l) / n_ar, index=idx)
    f["aroon_up_25"] = aroon_up
    f["aroon_down_25"] = aroon_dn
    f["aroon_osc_25"] = aroon_up - aroon_dn

    f["bop"] = (c - o) / (h - l)
    su = gain.rolling(14, min_periods=14).sum()
    sd = loss.rolling(14, min_periods=14).sum()
    f["cmo_14"] = 100.0 * (su - sd) / (su + sd)
    ema12 = _ema(c, 12)
    ema26 = _ema(c, 26)
    f["ppo"] = 100.0 * (ema12 - ema26) / ema26
    f["momentum_10_atr"] = (c - c.shift(10)) / atr14

    # ---- MACD family (normalised by price) ----
    macd = ema12 - ema26
    macd_sig = _ema(macd, 9)
    macd_hist = macd - macd_sig
    f["macd_pct"] = macd / c * 100.0
    f["macd_signal_pct"] = macd_sig / c * 100.0
    f["macd_hist_pct"] = macd_hist / c * 100.0
    f["macd_hist_chg"] = macd_hist.diff(1) / c * 100.0

    # ---- volatility ----
    f["natr_14"] = atr14 / c * 100.0
    f["tr_pct"] = tr / c * 100.0
    f["std_20_pct"] = c.rolling(20, min_periods=20).std() / c
    bb_mid = _sma(c, 20)
    bb_sd = c.rolling(20, min_periods=20).std(ddof=0)
    bb_up = bb_mid + 2.0 * bb_sd
    bb_lo = bb_mid - 2.0 * bb_sd
    f["bb_upper_rel"] = bb_up / c - 1.0
    f["bb_lower_rel"] = bb_lo / c - 1.0
    f["bb_middle_rel"] = bb_mid / c - 1.0
    f["bb_width"] = (bb_up - bb_lo) / bb_mid
    f["bb_position"] = (c - bb_lo) / (bb_up - bb_lo)

    # ---- volume indicators ----
    obv = (np.sign(delta).fillna(0.0) * v).cumsum()
    vsum10 = v.rolling(10, min_periods=10).sum()
    f["obv_roc_10"] = (obv - obv.shift(10)) / vsum10
    clv = (((c - l) - (h - c)) / (h - l)).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    ad = (clv * v).cumsum()
    f["ad_slope_10"] = (ad - ad.shift(10)) / vsum10
    f["adosc_norm"] = (_ema(ad, 3) - _ema(ad, 10)) / v.rolling(10, min_periods=10).mean()
    f["volume_change"] = v.pct_change(1).clip(-1.0, 10.0)
    f["rel_volume_5"] = v / _sma(v, 5)
    f["rel_volume_20"] = v / _sma(v, 20)

    # ---- price-transform features (relative to Close) ----
    f["avg_price_rel"] = (o + h + l + c) / 4.0 / c - 1.0
    f["median_price_rel"] = (h + l) / 2.0 / c - 1.0
    f["typical_price_rel"] = tp / c - 1.0
    f["weighted_close_rel"] = (h + l + 2.0 * c) / 4.0 / c - 1.0

    # ---- optional Hilbert-Transform cycle features (only if TA-Lib is installed) ----
    if _talib is not None:
        try:
            cv = c.values.astype(float)
            ht = {}
            ht["ht_trendmode"] = _talib.HT_TRENDMODE(cv).astype(float)
            ht["ht_dcperiod"] = _talib.HT_DCPERIOD(cv)
            ht["ht_dcphase"] = _talib.HT_DCPHASE(cv)
            inph, quad = _talib.HT_PHASOR(cv)
            ht["ht_phasor_inphase"] = inph / cv
            ht["ht_phasor_quadrature"] = quad / cv
            sine, lead = _talib.HT_SINE(cv)
            ht["ht_sine"] = sine
            ht["ht_leadsine"] = lead
            for k_, v_ in ht.items():
                f[k_] = pd.Series(v_, index=idx)
        except Exception:
            pass

    out = pd.DataFrame(f, index=idx)
    out = out.replace([np.inf, -np.inf], np.nan).ffill()
    return out.astype("float32")

def compute_features(df, symbol_col="Symbol"):
    # df must be sorted by [symbol_col, Date] with a unique index
    parts = []
    for _, g in df.groupby(symbol_col, sort=False):
        parts.append(build_features_for_stock(g))
    return pd.concat(parts).reindex(df.index)
'''

exec(FEATURE_CODE, globals())
(PROJECT_DIR / "feature_engineering.py").write_text(FEATURE_CODE.lstrip("\n"))
print("Feature code executed and saved to", PROJECT_DIR / "feature_engineering.py")

## 7. Target creation, per-stock features, and dataset assembly
* **Target:** `future_return = Close[t+1] / Close[t] - 1`;  `UP = 1` if `future_return > 0`, else `DOWN = 0` (an exactly flat day counts as DOWN). The last row of every stock has no label and is kept only as the "latest" row for live prediction - it never enters training/validation/test.
* Features are computed per stock on the complete history (they only look backwards). Leading warm-up rows (e.g. SMA-100) are dropped; sporadic gaps from zero-volume days are forward-filled (past values only).

In [ ]:
df_u = df_u.sort_values(["Symbol", "Date"]).reset_index(drop=True)

# ---- target (the ONLY place where the future is used) ----
df_u["future_close"] = df_u.groupby("Symbol", sort=False)["Close"].shift(-1)
df_u["future_return"] = df_u["future_close"] / df_u["Close"] - 1.0
df_u["target"] = (df_u["future_return"] > 0).astype(float).where(df_u["future_close"].notna())

t0 = time.time()
feat = compute_features(df_u)
print(f"Technical features computed for {df_u['Symbol'].nunique()} stocks in {time.time() - t0:.1f}s")

nan_share = feat.isna().mean()
FEATURES = [c for c in feat.columns if nan_share[c] < 0.5 and feat[c].nunique() > 1]
dropped_cols = [c for c in feat.columns if c not in FEATURES]
if dropped_cols:
    print("Dropped unusable feature columns:", dropped_cols)

full = pd.concat([df_u[["Symbol", "Date", "Open", "High", "Low", "Close", "Volume", "future_return", "target"]],
                  feat[FEATURES]], axis=1)

# keep each stock only from its first fully valid row onward (contiguous history, needed for sequences)
valid_row = full[FEATURES].notna().all(axis=1)
keep = valid_row.astype(int).groupby(full["Symbol"]).cummax().astype(bool)
rows_before = len(full)
full = full[keep].reset_index(drop=True)
remaining_nan = int(full[FEATURES].isna().sum().sum())
if remaining_nan:
    print(f"WARNING: {remaining_nan} isolated NaN feature values remain after warm-up; filled with 0.")
    full[FEATURES] = full[FEATURES].fillna(0.0)
full["pos"] = full.groupby("Symbol").cumcount()          # position inside the stock's valid history

print(f"Warm-up rows removed        : {rows_before - len(full):,}")
print(f"Rows in modelling frame     : {len(full):,}")
print(f"ACTUAL technical feature count generated: {len(FEATURES)}")
print("Feature names:"); print(", ".join(FEATURES))

## 8. Universal chronological split - performed **separately for every stock**
For each stock: first 70 % of its labelled rows = train, next 15 % = validation, last 15 % = test (never random). `EMBARGO_ROWS` rows at the end of train and validation are purged so that no label (which looks one day ahead) straddles a segment boundary. The three per-stock pieces are then concatenated into the **universal** train / validation / test sets.

The test indices are kept behind `open_test()`. That function is called exactly once, after all models are frozen, and logs the moment it happens (this log feeds the leakage audit).

In [ ]:
lab = full["target"].notna()
n_lab = lab.groupby(full["Symbol"]).transform("sum").astype(int)
n_tr = np.floor(TRAIN_FRAC * n_lab).astype(int)
n_va = np.floor(VAL_FRAC * n_lab).astype(int)
p = full["pos"]

split = np.where(~lab, "latest",
        np.where(p < n_tr, "train",
        np.where(p < n_tr + n_va, "val", "test"))).astype(object)
purge = lab & (((p >= n_tr - EMBARGO_ROWS) & (p < n_tr)) |
               ((p >= n_tr + n_va - EMBARGO_ROWS) & (p < n_tr + n_va)))
split[purge.values] = "purge"
full["split"] = split

# drop stocks whose segments are too short to be useful
cnt = (full.groupby(["Symbol", "split"]).size().unstack(fill_value=0)
           .reindex(columns=["train", "val", "test", "purge", "latest"], fill_value=0))
bad = cnt[(cnt["train"] < WINDOW + 100) | (cnt["val"] < 30) | (cnt["test"] < 30)].index.tolist()
if bad:
    print(f"Dropping {len(bad)} stocks with too little data in a segment: {bad[:15]}{'...' if len(bad) > 15 else ''}")
    if TARGET_SYMBOL in bad:
        raise RuntimeError(f"{TARGET_SYMBOL} does not have enough history for a train/val/test split.")
    full = full[~full["Symbol"].isin(bad)].reset_index(drop=True)
UNIVERSE = [s for s in UNIVERSE if s not in bad]

SPLIT = full["split"].values
tr_idx = np.flatnonzero(SPLIT == "train")
va_idx = np.flatnonzero(SPLIT == "val")
_TEST_IDX = np.flatnonzero(SPLIT == "test")          # private: only reachable through open_test()
def open_test(reason):
    log_event("test_opened", reason=reason)
    return _TEST_IDX.copy()

y_all = full["target"].values.astype(np.float32)       # NaN only on 'latest' rows
X_all = full[FEATURES].values.astype(np.float32)
FEAT_INDEX = {f: i for i, f in enumerate(FEATURES)}

# ---- per-stock split dates ----
rows = []
for sym, g in full.groupby("Symbol", sort=False):
    r = {"Symbol": sym, "total_rows": len(g)}
    for s in ["train", "val", "test"]:
        gs = g[g["split"] == s]
        r[f"{s}_rows"] = len(gs)
        r[f"{s}_start"] = gs["Date"].min().date()
        r[f"{s}_end"] = gs["Date"].max().date()
    rows.append(r)
split_table = pd.DataFrame(rows)

print(f"Stocks in the universal set : {full['Symbol'].nunique()}")
print(f"Universal TRAIN rows        : {len(tr_idx):,}")
print(f"Universal VALIDATION rows   : {len(va_idx):,}")
print(f"Universal TEST rows         : {len(_TEST_IDX):,}   (untouched until the final evaluation)")
print(f"Purged boundary rows        : {int((SPLIT == 'purge').sum()):,}")
print("\nActual chronological split dates for EVERY stock:")
print(split_table.to_string(index=False))

## 9. Utilities (metrics, threshold selection, XGBoost factory)
Decision thresholds are chosen **on the validation set only** (maximising balanced accuracy over a 0.40-0.60 grid, or fixed at 0.5 if `THRESHOLD_MODE="fixed_0.5"`) and then frozen *before* the test set is opened. ROC-AUC does not depend on the threshold.

In [ ]:
def compute_metrics(y_true, prob, thr):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    auc = roc_auc_score(y_true, prob) if len(np.unique(y_true)) == 2 else np.nan
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": auc,
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        "sensitivity": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
    }

def pick_threshold(y_val, p_val):
    if THRESHOLD_MODE == "fixed_0.5":
        return 0.5
    grid = np.round(np.arange(0.40, 0.6001, 0.005), 3)
    best = min(grid, key=lambda t: (-balanced_accuracy_score(y_val, (p_val >= t).astype(int)), abs(t - 0.5)))
    return float(best)

def make_xgb(**override):
    params = dict(XGB_PARAMS); params.update(override)
    return XGBClassifier(**params)

def fmt(x, nd=4):
    return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.{nd}f}"

## 10. Exploratory visuals: price history and class balance

In [ ]:
# 1) historical price of the target stock (split regions shaded; no labels or predictions involved)
tg = full[full["Symbol"] == TARGET_SYMBOL]
trow = split_table[split_table["Symbol"] == TARGET_SYMBOL].iloc[0]
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(tg["Date"], tg["Close"], color="#1f4e79", lw=1.2, label="Close")
ax.plot(tg["Date"], tg["Close"].rolling(50).mean(), color="#e69f00", lw=1, label="SMA 50")
ax.plot(tg["Date"], tg["Close"].rolling(200).mean(), color="#009e73", lw=1, label="SMA 200")
for name, col in [("train", "#cfe2f3"), ("val", "#fff2cc"), ("test", "#f4cccc")]:
    ax.axvspan(pd.Timestamp(trow[f"{name}_start"]), pd.Timestamp(trow[f"{name}_end"]), color=col, alpha=0.6, label=f"{name} period")
ax.set_title(f"{TARGET_SYMBOL}: historical close price with chronological split regions")
ax.set_ylabel("Close (INR)"); ax.legend(ncol=6, fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(PROJECT_DIR / "historical_price_target.png", dpi=150); plt.show()

# 2) class distribution across the universal TRAINING data
ytr = y_all[tr_idx].astype(int); yva = y_all[va_idx].astype(int)
counts = np.bincount(ytr, minlength=2)
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["DOWN (0)", "UP (1)"], counts, color=["#d55e00", "#009e73"])
for b, c_ in zip(bars, counts):
    ax.text(b.get_x() + b.get_width() / 2, c_, f"{c_:,}\n({c_ / counts.sum():.1%})", ha="center", va="bottom", fontsize=9)
ax.set_title("Class distribution - universal training set"); ax.set_ylabel("rows")
plt.tight_layout(); plt.savefig(PROJECT_DIR / "class_distribution_train.png", dpi=150); plt.show()
print(f"Train UP share: {ytr.mean():.4f} | Validation UP share: {yva.mean():.4f}")

## 11. Experiments 1-3 (+ majority baseline): trained on TRAIN, judged on VALIDATION only
* **Majority baseline** - always predicts the training-set majority class.
* **Experiment 1 - OHLCV XGBoost:** `Open, High, Low, Close, Volume` + 1-day return.
* **Experiment 2 - Expanded XGBoost:** all technical features.

All use the same transparent XGBoost configuration (`n_estimators=100`). Test scores are computed later, in one go, once everything is frozen.

In [ ]:
OHLCV_FEATURES = ["Open", "High", "Low", "Close", "Volume", "ret_1d"]
X_ohlcv = full[OHLCV_FEATURES].values.astype(np.float32)
BASES = {"ohlcv": X_ohlcv, "all": X_all}
MODELS = {}

def entry_X(entry, idx):
    return BASES[entry["base"]][idx][:, entry["cols"]]

def register_model(name, model, base, cols, n_features, extra=None):
    e = {"name": name, "model": model, "base": base, "cols": np.asarray(cols), "n_features": n_features}
    e.update(extra or {})
    p_va = model.predict_proba(entry_X(e, va_idx))[:, 1]
    e["val_prob"] = p_va
    e["thr"] = pick_threshold(yva, p_va)
    e["val_metrics"] = compute_metrics(yva, p_va, e["thr"])
    MODELS[name] = e
    register_use("threshold", SPLIT, va_idx)
    return e

# ---- majority baseline ----
maj_class = int(ytr.mean() >= 0.5)
maj_prob_va = np.full(len(va_idx), ytr.mean())
MODELS["Majority Baseline"] = {"name": "Majority Baseline", "model": None, "n_features": 0, "maj_class": maj_class,
                               "thr": 0.0 if maj_class == 1 else 1.01, "val_prob": maj_prob_va,
                               "train_up_rate": float(ytr.mean())}
MODELS["Majority Baseline"]["val_metrics"] = compute_metrics(yva, maj_prob_va, MODELS["Majority Baseline"]["thr"])

# ---- Experiment 1: OHLCV ----
register_use("xgb_fit", SPLIT, tr_idx)
m1 = make_xgb(); m1.fit(X_ohlcv[tr_idx], ytr)
register_model("OHLCV XGBoost", m1, "ohlcv", np.arange(len(OHLCV_FEATURES)), len(OHLCV_FEATURES))

# ---- Experiment 2: expanded technical features ----
m2 = make_xgb(); m2.fit(X_all[tr_idx], ytr)
register_model("Expanded Feature XGBoost", m2, "all", np.arange(len(FEATURES)), len(FEATURES))

for k in ["Majority Baseline", "OHLCV XGBoost", "Expanded Feature XGBoost"]:
    vm = MODELS[k]["val_metrics"]
    print(f"{k:28s} | features {MODELS[k]['n_features']:3d} | threshold {MODELS[k]['thr']:.3f} | "
          f"VAL acc {vm['accuracy']:.4f} | bal.acc {vm['balanced_accuracy']:.4f} | F1 {vm['f1']:.4f} | AUC {fmt(vm['roc_auc'])}")
log_event("baseline_models_trained")

## 12. Experiment 3 - Genetic Algorithm feature selection (main research method)
* **Chromosome:** binary vector, `1` = feature selected, `0` = not selected.
* **Fitness (paper-style baseline):** train a *small* XGBoost on the TRAINING sample with the selected features → **classification accuracy on the VALIDATION sample**. No feature-count penalty, no test data, no ROC-AUC.
* **Operators:** random initialisation, tournament selection (k = 3), uniform crossover (applied with probability 0.50), mutation (a chromosome mutates with probability 0.375; ~3 genes flip), elitism = 1, best-chromosome tracking. A minimum of 5 features is enforced. A fitness cache avoids re-training identical chromosomes.
* **Compute control:** the GA fitness uses stratified samples of the *training* and *validation* sets (never test); the exact row counts are printed below. `QUICK_MODE` shrinks the GA.
* Note: the GA's best validation accuracy is an **optimistic** number (the GA picked the subset that did best on that very data). Only the final test result is an honest estimate.

In [ ]:
def stratified_sample(idx_arr, n, seed):
    if n >= len(idx_arr):
        return np.sort(idx_arr)
    sel, _ = train_test_split(idx_arr, train_size=int(n), stratify=y_all[idx_arr].astype(int), random_state=seed)
    return np.sort(sel)

ga_tr_idx = stratified_sample(tr_idx, GA_TRAIN_ROWS, SEED)
ga_va_idx = stratified_sample(va_idx, GA_VAL_ROWS, SEED + 1)
register_use("ga_fit", SPLIT, ga_tr_idx)
register_use("ga_fitness_val", SPLIT, ga_va_idx)
print(f"GA fitness data -> TRAIN rows used: {len(ga_tr_idx):,} of {len(tr_idx):,} | VALIDATION rows used: {len(ga_va_idx):,} of {len(va_idx):,} | test rows used: 0")

Xga_tr, yga_tr = X_all[ga_tr_idx], y_all[ga_tr_idx].astype(int)
Xga_va, yga_va = X_all[ga_va_idx], y_all[ga_va_idx].astype(int)
GA_XGB_PARAMS = dict(XGB_PARAMS, n_estimators=GA_TREES, max_depth=4)

def ga_feature_selection(X_tr, y_tr, X_va, y_va, names, pop_size, generations, p_cx, p_mut,
                         elitism, tournament_k, min_features, flip_count, seed, xgb_params):
    rng = np.random.default_rng(seed)
    n = len(names)
    gene_flip = flip_count / n
    cache = {}

    def fitness(mask):
        key = mask.tobytes()
        if key not in cache:
            cols = np.flatnonzero(mask)
            m = XGBClassifier(**xgb_params)
            m.fit(X_tr[:, cols], y_tr)
            cache[key] = float(accuracy_score(y_va, m.predict(X_va[:, cols])))   # FITNESS = VALIDATION ACCURACY
        return cache[key]

    def repair(mask):
        k = int(mask.sum())
        if k < min_features:
            off = np.flatnonzero(~mask)
            mask[rng.choice(off, size=min_features - k, replace=False)] = True
        return mask

    def tournament(pop, fits):
        cand = rng.choice(len(pop), size=min(tournament_k, len(pop)), replace=False)
        return pop[cand[np.argmax(fits[cand])]]

    pop = np.array([repair(rng.random(n) < 0.5) for _ in range(pop_size)])
    history, best_mask, best_fit = [], None, -1.0
    t_start = time.time()
    for gen in range(1, generations + 1):
        fits = np.array([fitness(ind) for ind in pop])
        order = np.argsort(-fits, kind="stable")
        if fits[order[0]] > best_fit:
            best_fit, best_mask = float(fits[order[0]]), pop[order[0]].copy()
        history.append({"generation": gen, "best_val_accuracy": float(fits.max()), "avg_val_accuracy": float(fits.mean()),
                        "n_selected_features_best": int(pop[order[0]].sum()),
                        "best_so_far_val_accuracy": best_fit, "n_selected_features_best_so_far": int(best_mask.sum()),
                        "unique_chromosomes_evaluated": len(cache)})
        print(f"Generation {gen:02d}/{generations} | best val acc {fits.max():.4f} | avg val acc {fits.mean():.4f} | "
              f"selected features (best) {int(pop[order[0]].sum()):3d} | elapsed {time.time() - t_start:6.1f}s")
        if gen == generations:
            break
        new_pop = [pop[order[i]].copy() for i in range(elitism)]            # elitism
        while len(new_pop) < pop_size:
            p1, p2 = tournament(pop, fits), tournament(pop, fits)
            c1, c2 = p1.copy(), p2.copy()
            if rng.random() < p_cx:                                         # uniform crossover
                swap = rng.random(n) < 0.5
                c1[swap], c2[swap] = p2[swap], p1[swap]
            for c in (c1, c2):
                if rng.random() < p_mut:                                    # mutation
                    flip = rng.random(n) < gene_flip
                    c[flip] = ~c[flip]
                new_pop.append(repair(c))
        pop = np.array(new_pop[:pop_size])
    return best_mask, best_fit, pd.DataFrame(history)

best_mask, ga_best_fit, ga_history = ga_feature_selection(
    Xga_tr, yga_tr, Xga_va, yga_va, FEATURES, GA_POP, GA_GENS, GA_CROSSOVER, GA_MUTATION,
    GA_ELITISM, GA_TOURNAMENT_K, GA_MIN_FEATURES, GA_GENE_FLIP_COUNT, SEED, GA_XGB_PARAMS)

sel_idx = np.flatnonzero(best_mask)
SELECTED = [FEATURES[i] for i in sel_idx]
log_event("ga_done", best_validation_accuracy=ga_best_fit, n_selected=len(SELECTED))
print(f"\nGA finished. Best VALIDATION accuracy (GA sample): {ga_best_fit:.4f}")
print(f"Features before GA: {len(FEATURES)} | after GA: {len(SELECTED)}")
print("Selected features:", SELECTED)

## 13. Experiment 4 - final GA-XGBoost (+ optional separate tuning experiment)
* The best chromosome's features are used to train the final `XGBClassifier` on the **full universal training set** (`n_estimators=100`, parameters frozen).
* `GA-selected features (GA-sample XGB)` in the ablation = the small XGBoost *as used inside the GA* (fewer trees, trained on the GA sample) evaluated with the GA subset - it isolates the value of the subset from the value of the larger final model.
* The optional tuning experiment is a small random search over XGBoost hyper-parameters using **train → validation only**. It is reported as a *separate* row; the headline "GA-XGBoost" stays the transparent paper-style configuration.

In [ ]:
# ---- final paper-style GA-XGBoost ----
register_use("xgb_fit", SPLIT, tr_idx)
ga_model = make_xgb()
ga_model.fit(X_all[tr_idx][:, sel_idx], ytr)
register_model("GA-XGBoost", ga_model, "all", sel_idx, len(SELECTED))

# ---- GA-selected features evaluated with the small in-GA model (ablation row) ----
ga_small = XGBClassifier(**GA_XGB_PARAMS)
ga_small.fit(Xga_tr[:, sel_idx], yga_tr)
register_model("GA-selected features (GA-sample XGB)", ga_small, "all", sel_idx, len(SELECTED))

# ---- feature importance of the final GA-XGBoost ----
fi = pd.DataFrame({"Feature": SELECTED, "Importance": ga_model.feature_importances_}).sort_values("Importance", ascending=False).reset_index(drop=True)
print("Top 20 selected features (XGBoost importance):")
display(fi.head(20))

# ---- optional hyper-parameter tuning (train/validation only) ----
tuning_log = pd.DataFrame()
if RUN_TUNING:
    register_use("tuning_fit", SPLIT, tr_idx)
    register_use("tuning_eval", SPLIT, va_idx)
    space = dict(max_depth=[2, 3, 4, 5, 6, 8], learning_rate=[0.01, 0.03, 0.05, 0.1], n_estimators=[100, 200, 300, 500],
                 subsample=[0.6, 0.8, 1.0], colsample_bytree=[0.5, 0.7, 0.9, 1.0], min_child_weight=[1, 5, 10, 20],
                 reg_lambda=[1, 5, 10], gamma=[0, 1, 5])
    rng_t = np.random.default_rng(SEED)
    Xtr_sel, Xva_sel = X_all[tr_idx][:, sel_idx], X_all[va_idx][:, sel_idx]
    recs, best_t = [], None
    for it in range(TUNE_ITERS):
        cfg = {k: v[int(rng_t.integers(len(v)))] for k, v in space.items()}
        m = make_xgb(**cfg); m.fit(Xtr_sel, ytr)
        pv = m.predict_proba(Xva_sel)[:, 1]
        acc = accuracy_score(yva, (pv >= 0.5).astype(int))
        recs.append({**cfg, "val_accuracy": acc})
        if best_t is None or acc > best_t[0]:
            best_t = (acc, cfg, m)
        print(f"Tuning {it + 1:02d}/{TUNE_ITERS} | val acc {acc:.4f} | {cfg}")
    tuning_log = pd.DataFrame(recs).sort_values("val_accuracy", ascending=False)
    register_model("GA-XGBoost (tuned)", best_t[2], "all", sel_idx, len(SELECTED), extra={"tuned_params": best_t[1]})
    print("Best tuned config (by validation accuracy):", best_t[1])
    log_event("tuning_done", best_val_accuracy=float(best_t[0]))
else:
    print("RUN_TUNING=False -> hyper-parameter tuning skipped.")
    log_event("tuning_skipped")

if RUN_SHAP:
    try:
        import shap
        rs = np.random.default_rng(SEED).choice(va_idx, size=min(2000, len(va_idx)), replace=False)
        sv = shap.TreeExplainer(ga_model).shap_values(X_all[rs][:, sel_idx])
        shap.summary_plot(sv, X_all[rs][:, sel_idx], feature_names=SELECTED, show=False)
        plt.tight_layout(); plt.savefig(PROJECT_DIR / "shap_summary.png", dpi=150); plt.show()
    except Exception as e:
        print("SHAP skipped:", e)

for k in ["GA-XGBoost", "GA-selected features (GA-sample XGB)"] + (["GA-XGBoost (tuned)"] if RUN_TUNING else []):
    vm = MODELS[k]["val_metrics"]
    print(f"{k:38s} | features {MODELS[k]['n_features']:3d} | thr {MODELS[k]['thr']:.3f} | VAL acc {vm['accuracy']:.4f} | AUC {fmt(vm['roc_auc'])}")
log_event("xgb_final_fit")

## 14. Experiment 5 - ONE universal LSTM
* A single LSTM is trained across all stocks (no per-company models). Architecture exactly as specified: `LSTM(64, return_sequences) → Dropout(0.2) → LSTM(32) → Dropout(0.2) → Dense(16, relu) → Dense(1, sigmoid)`; Adam, binary cross-entropy, metrics accuracy + AUC, `EarlyStopping`, `shuffle=False`.
* Inputs: the GA-selected (stationary, relative) features, standardised with a `StandardScaler` **fitted on the universal training rows only** (then clipped to ±5).
* **Sequences never cross stocks:** a sample ending at row *t* is allowed only if `pos[t] >= 59`, i.e. all 60 rows belong to the same stock's contiguous history. Windows are assembled on the fly from index arrays (no giant 3-D array). Samples are ordered by date (then symbol) so that un-shuffled batches still mix stocks.
* A window may look back into earlier segments (e.g. the first test sample uses validation-period *features*); that is legitimate because it is past information and labels are never used as inputs.

In [ ]:
LSTM_FEATURES = list(SELECTED)
X_l_raw = full[LSTM_FEATURES].values.astype(np.float32)

lstm_scaler = StandardScaler().fit(X_l_raw[tr_idx])            # TRAIN rows only
register_use("scaler_fit", SPLIT, tr_idx)
X_l = np.clip(lstm_scaler.transform(X_l_raw), -CLIP_Z, CLIP_Z).astype(np.float32)

sym_codes = full["Symbol"].astype("category").cat.codes.values
pos_arr = full["pos"].values
date_int = full["Date"].values.astype("datetime64[ns]").astype("int64")

def window_end_indices(split_idx):
    e = split_idx[pos_arr[split_idx] >= WINDOW - 1]              # whole window inside ONE stock's history
    order = np.lexsort((sym_codes[e], date_int[e]))              # by date, then symbol
    return e[order]

class WindowSequence(tf.keras.utils.Sequence):
    def __init__(self, X, y, end_idx, window, batch_size, **kwargs):
        super().__init__(**kwargs)
        self.X, self.y, self.end_idx = X, y, end_idx
        self.offsets = np.arange(-(window - 1), 1)
        self.batch_size = batch_size
    def __len__(self):
        return int(math.ceil(len(self.end_idx) / self.batch_size))
    def __getitem__(self, i):
        e = self.end_idx[i * self.batch_size:(i + 1) * self.batch_size]
        return self.X[e[:, None] + self.offsets[None, :]], self.y[e].reshape(-1, 1)

lstm_tr_end = window_end_indices(tr_idx)
lstm_va_end = window_end_indices(va_idx)
register_use("lstm_train", SPLIT, lstm_tr_end)
register_use("lstm_val", SPLIT, lstm_va_end)
train_seq = WindowSequence(X_l, y_all, lstm_tr_end, WINDOW, LSTM_BATCH)
val_seq = WindowSequence(X_l, y_all, lstm_va_end, WINDOW, LSTM_BATCH)
print(f"LSTM input features: {len(LSTM_FEATURES)} | window: {WINDOW} days")
print(f"LSTM train sequences: {len(lstm_tr_end):,} | validation sequences: {len(lstm_va_end):,}")

def build_lstm(n_features):
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(WINDOW, n_features)),
        tf.keras.layers.LSTM(64, return_sequences=True),
        tf.keras.layers.Dropout(0.20),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dropout(0.20),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return m

lstm_model = build_lstm(len(LSTM_FEATURES))
lstm_model.summary()
history = lstm_model.fit(
    train_seq, validation_data=val_seq, epochs=LSTM_EPOCHS, shuffle=False, verbose=2,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=LSTM_PATIENCE, restore_best_weights=True)])
lstm_hist = {k: [float(x) for x in v] for k, v in history.history.items()}

p_va_lstm = lstm_model.predict(val_seq, verbose=0).ravel()
y_va_lstm = y_all[lstm_va_end].astype(int)
lstm_thr = pick_threshold(y_va_lstm, p_va_lstm)
register_use("threshold", SPLIT, lstm_va_end)
lstm_val_metrics = compute_metrics(y_va_lstm, p_va_lstm, lstm_thr)
MODELS["Universal LSTM"] = {"name": "Universal LSTM", "model": lstm_model, "n_features": len(LSTM_FEATURES), "thr": lstm_thr,
                            "val_prob": p_va_lstm, "val_metrics": lstm_val_metrics}
print(f"\nLSTM VALIDATION | threshold {lstm_thr:.3f} | acc {lstm_val_metrics['accuracy']:.4f} | AUC {fmt(lstm_val_metrics['roc_auc'])} | F1 {lstm_val_metrics['f1']:.4f}")
log_event("lstm_fit")

## 15. FINAL EVALUATION - the untouched test set is opened here, once
Everything is now frozen: features, GA subset, XGBoost parameters, LSTM weights, scaler and decision thresholds. No model, threshold or feature is changed after this point. Validation scores shown next to test scores are for context only - **the headline result is always the test column.**

In [ ]:
log_event("models_frozen", thresholds={k: v["thr"] for k, v in MODELS.items()})
TEST_IDX = open_test("final evaluation of all frozen models (opened once)")
register_use("final_eval", SPLIT, TEST_IDX)
assert not np.isnan(y_all[TEST_IDX]).any(), "test rows must all be labelled"
yte = y_all[TEST_IDX].astype(int)
print(f"Universal TEST set: {len(TEST_IDX):,} rows from {full.iloc[TEST_IDX]['Symbol'].nunique()} stocks | UP share {yte.mean():.4f}")

lstm_te_end = window_end_indices(TEST_IDX)
test_seq = WindowSequence(X_l, y_all, lstm_te_end, WINDOW, LSTM_BATCH)

for name, e in MODELS.items():
    if name == "Majority Baseline":
        e["test_eval_idx"], e["test_prob"] = TEST_IDX, np.full(len(TEST_IDX), e["train_up_rate"])
    elif name == "Universal LSTM":
        e["test_eval_idx"], e["test_prob"] = lstm_te_end, e["model"].predict(test_seq, verbose=0).ravel()
    else:
        e["test_eval_idx"], e["test_prob"] = TEST_IDX, e["model"].predict_proba(entry_X(e, TEST_IDX))[:, 1]
    e["test_metrics"] = compute_metrics(y_all[e["test_eval_idx"]].astype(int), e["test_prob"], e["thr"])

if len(lstm_te_end) != len(TEST_IDX):
    print(f"NOTE: LSTM can score {len(lstm_te_end):,} of {len(TEST_IDX):,} test rows (needs a full {WINDOW}-day window).")
else:
    print(f"LSTM and tree models are scored on the identical {len(TEST_IDX):,} test rows.")

MODEL_ORDER = ["Majority Baseline", "OHLCV XGBoost", "Expanded Feature XGBoost", "GA-XGBoost"] + \
              (["GA-XGBoost (tuned)"] if "GA-XGBoost (tuned)" in MODELS else []) + ["Universal LSTM"]
comp = pd.DataFrame([{
    "Model": n, "Feature Count": MODELS[n]["n_features"],
    "Validation Accuracy": MODELS[n]["val_metrics"]["accuracy"], "Test Accuracy": MODELS[n]["test_metrics"]["accuracy"],
    "Precision": MODELS[n]["test_metrics"]["precision"], "Recall": MODELS[n]["test_metrics"]["recall"],
    "F1": MODELS[n]["test_metrics"]["f1"], "ROC-AUC": MODELS[n]["test_metrics"]["roc_auc"],
    "Test Balanced Acc": MODELS[n]["test_metrics"]["balanced_accuracy"], "Threshold": MODELS[n]["thr"]} for n in MODEL_ORDER])

print("\n" + "=" * 100); print("FINAL MODEL COMPARISON (test columns = untouched universal test set)"); print("=" * 100)
display(comp.round(4))
print(f"Reference: share of UP days in the test set = {yte.mean():.4f}  (an 'always UP' predictor would score this)")

print("\nTest-set confusion counts and extra metrics:")
extra = pd.DataFrame([{"Model": n, **{k: MODELS[n]["test_metrics"][k] for k in ["TP", "TN", "FP", "FN", "specificity", "sensitivity", "balanced_accuracy"]}} for n in MODEL_ORDER])
display(extra.round(4))
for n in ["GA-XGBoost", "Universal LSTM"]:
    e = MODELS[n]
    print(f"\nClassification report - {n} (test, threshold {e['thr']:.3f})")
    print(classification_report(y_all[e["test_eval_idx"]].astype(int), (e["test_prob"] >= e["thr"]).astype(int),
                                target_names=["DOWN", "UP"], zero_division=0, digits=4))

ablation = pd.DataFrame([
    {"Configuration": lab, "Validation Accuracy": MODELS[k]["val_metrics"]["accuracy"], "Test Accuracy": MODELS[k]["test_metrics"]["accuracy"],
     "Test F1": MODELS[k]["test_metrics"]["f1"], "Test ROC-AUC": MODELS[k]["test_metrics"]["roc_auc"]}
    for lab, k in [("OHLCV", "OHLCV XGBoost"), ("Expanded technical indicators", "Expanded Feature XGBoost"),
                   ("GA-selected features (GA-sample XGB)", "GA-selected features (GA-sample XGB)"),
                   ("GA-XGBoost", "GA-XGBoost"), ("LSTM", "Universal LSTM")]])
print("\nABLATION STUDY"); display(ablation.round(4))

# ---------- per-stock evaluation ----------
xe, le = MODELS["GA-XGBoost"], MODELS["Universal LSTM"]
p_lstm_full = np.full(len(full), np.nan)
p_lstm_full[le["test_eval_idx"]] = le["test_prob"]
pred_df = pd.DataFrame({
    "Symbol": full["Symbol"].values[TEST_IDX], "Date": full["Date"].values[TEST_IDX], "Close": full["Close"].values[TEST_IDX],
    "y_true": yte, "xgb_prob_up": xe["test_prob"], "lstm_prob_up": p_lstm_full[TEST_IDX]})
pred_df["xgb_pred"] = (pred_df["xgb_prob_up"] >= xe["thr"]).astype(int)
pred_df["lstm_pred"] = np.where(pred_df["lstm_prob_up"].isna(), np.nan, (pred_df["lstm_prob_up"] >= le["thr"]).astype(float))

ps = []
for sym, g in pred_df.groupby("Symbol"):
    m = compute_metrics(g["y_true"], g["xgb_prob_up"], xe["thr"])
    r = {"Stock": sym, "Number of rows": int(rows_per_stock[sym]), "Test rows": len(g), "Accuracy": m["accuracy"],
         "Precision": m["precision"], "Recall": m["recall"], "F1": m["f1"], "ROC-AUC": m["roc_auc"], "Test UP share": g["y_true"].mean()}
    gl = g.dropna(subset=["lstm_prob_up"])
    if len(gl):
        ml = compute_metrics(gl["y_true"], gl["lstm_prob_up"], le["thr"])
        r["LSTM Accuracy"], r["LSTM ROC-AUC"] = ml["accuracy"], ml["roc_auc"]
    ps.append(r)
per_stock = pd.DataFrame(ps).sort_values("Accuracy", ascending=False).reset_index(drop=True)

acc_s = per_stock["Accuracy"]
stock_stats = {"mean": float(acc_s.mean()), "median": float(acc_s.median()), "std": float(acc_s.std(ddof=0)),
               "min": float(acc_s.min()), "max": float(acc_s.max())}
print(f"\nPER-STOCK TEST RESULTS - universal GA-XGBoost applied to each stock's own untouched test period ({len(per_stock)} stocks)")
display(per_stock.round(4))
print("Accuracy across stocks -> " + " | ".join(f"{k}: {v:.4f}" for k, v in stock_stats.items()))
tg_row = per_stock[per_stock["Stock"] == TARGET_SYMBOL]
if len(tg_row):
    print(f"\n{TARGET_SYMBOL} (target) test accuracy: {tg_row['Accuracy'].iloc[0]:.4f} on {int(tg_row['Test rows'].iloc[0])} rows")

## 16. Visualisations
All figures are saved into the project folder (names as required for the report).

In [ ]:
def savefig(name):
    plt.tight_layout(); plt.savefig(PROJECT_DIR / name, dpi=150); plt.show()

PLOT_MODELS = [n for n in ["OHLCV XGBoost", "Expanded Feature XGBoost", "GA-XGBoost", "Universal LSTM"] if n in MODELS]
colors = {"OHLCV XGBoost": "#999999", "Expanded Feature XGBoost": "#e69f00", "GA-XGBoost": "#0072b2", "Universal LSTM": "#d55e00"}

# 3) GA convergence
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(ga_history["generation"], ga_history["best_val_accuracy"], "o-", color="#0072b2", label="best validation accuracy")
ax.plot(ga_history["generation"], ga_history["avg_val_accuracy"], "s--", color="#e69f00", label="average validation accuracy")
ax.set_xlabel("generation"); ax.set_ylabel("validation accuracy (GA fitness)"); ax.grid(alpha=0.3)
ax2 = ax.twinx(); ax2.plot(ga_history["generation"], ga_history["n_selected_features_best"], ":", color="#009e73", label="# features (best)")
ax2.set_ylabel("# selected features"); h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="lower right", fontsize=8); ax.set_title("GA convergence")
savefig("ga_convergence.png")

# 4) top feature importance
top = fi.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6)); ax.barh(top["Feature"], top["Importance"], color="#0072b2")
ax.set_title("GA-XGBoost: top 20 selected features (XGBoost importance)"); ax.set_xlabel("importance")
savefig("feature_importance.png")

# 5) confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, n in zip(axes, ["GA-XGBoost", "Universal LSTM"]):
    tm = MODELS[n]["test_metrics"]; cm = np.array([[tm["TN"], tm["FP"]], [tm["FN"], tm["TP"]]])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], ["pred DOWN", "pred UP"]); ax.set_yticks([0, 1], ["true DOWN", "true UP"])
    ax.set_title(f"{n}\ntest accuracy {tm['accuracy']:.3f}")
savefig("confusion_matrix.png")

# 6) ROC and 7) precision-recall on the test set
fig, ax = plt.subplots(figsize=(6, 5))
for n in PLOT_MODELS:
    yy = y_all[MODELS[n]["test_eval_idx"]].astype(int); fpr, tpr, _ = roc_curve(yy, MODELS[n]["test_prob"])
    ax.plot(fpr, tpr, color=colors[n], label=f"{n} (AUC {MODELS[n]['test_metrics']['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="chance"); ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
ax.set_title("ROC curves - universal test set"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
savefig("roc_curve.png")

fig, ax = plt.subplots(figsize=(6, 5))
for n in PLOT_MODELS:
    yy = y_all[MODELS[n]["test_eval_idx"]].astype(int); pr, rc, _ = precision_recall_curve(yy, MODELS[n]["test_prob"])
    ax.plot(rc, pr, color=colors[n], label=f"{n} (AP {average_precision_score(yy, MODELS[n]['test_prob']):.3f})")
ax.axhline(yte.mean(), color="k", ls="--", lw=0.8, label="UP base rate"); ax.set_xlabel("recall"); ax.set_ylabel("precision")
ax.set_title("Precision-recall curves - universal test set"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
savefig("precision_recall_curve.png")

# 8-11) LSTM training curves
ep = np.arange(1, len(lstm_hist["loss"]) + 1)
fig, ax = plt.subplots(figsize=(7, 4)); ax.plot(ep, lstm_hist["accuracy"], label="train accuracy"); ax.plot(ep, lstm_hist["val_accuracy"], label="validation accuracy")
ax.set_xlabel("epoch"); ax.set_ylabel("accuracy"); ax.set_title("Universal LSTM - accuracy"); ax.legend(); ax.grid(alpha=0.3)
savefig("lstm_training_accuracy.png")
fig, ax = plt.subplots(figsize=(7, 4)); ax.plot(ep, lstm_hist["loss"], label="train loss"); ax.plot(ep, lstm_hist["val_loss"], label="validation loss")
ax.set_xlabel("epoch"); ax.set_ylabel("binary cross-entropy"); ax.set_title("Universal LSTM - loss"); ax.legend(); ax.grid(alpha=0.3)
savefig("lstm_training_loss.png")

# 12-13) per-stock accuracy and ROC-AUC
def per_stock_bar(col, fname, title, ref=None):
    d = per_stock.sort_values(col, ascending=False)
    fig, ax = plt.subplots(figsize=(max(9, min(26, 0.25 * len(d))), 4.5))
    ax.bar(np.arange(len(d)), d[col], color=["#d55e00" if s == TARGET_SYMBOL else "#0072b2" for s in d["Stock"]])
    ax.axhline(d[col].mean(), color="k", ls="--", lw=1, label=f"mean {d[col].mean():.3f}")
    if ref is not None: ax.axhline(ref, color="gray", ls=":", lw=1, label=f"chance {ref}")
    if len(d) <= 40: ax.set_xticks(np.arange(len(d)), d["Stock"], rotation=90, fontsize=7)
    else: ax.set_xlabel(f"{len(d)} stocks (target stock highlighted in orange)")
    lo = max(0.0, d[col].min() - 0.05); ax.set_ylim(lo, min(1.0, d[col].max() + 0.05))
    ax.set_ylabel(col); ax.set_title(title); ax.legend(); ax.grid(alpha=0.3, axis="y")
    savefig(fname)
per_stock_bar("Accuracy", "per_stock_accuracy.png", "Universal GA-XGBoost: test accuracy per stock", ref=0.5)
per_stock_bar("ROC-AUC", "per_stock_roc_auc.png", "Universal GA-XGBoost: test ROC-AUC per stock", ref=0.5)

## 17. Evaluation Mode 2 (optional): leave-one-stock-out
**Mode 1** (the main pipeline above) = *within-stock chronological generalisation*: train on the early history of all stocks, test on the later history of the same stocks.

**Mode 2** = can the universal model handle a *company it never trained on*? For each held-out stock an XGBoost model (same parameters, GA-selected features) is trained on all *other* stocks' training rows, validation-checked on the other stocks' validation rows, and finally tested on the held-out stock's test period. Disclosure: the GA feature subset was chosen using all stocks (including the held-out one), so this checks generalisation of the learned weights, not of the feature selection. Disabled by default (`RUN_LOSO = False`) because it re-trains one model per held-out stock.

In [ ]:
loso_df = pd.DataFrame()
if RUN_LOSO:
    names = LOSO_STOCKS or [TARGET_SYMBOL] + [sym_map[norm_symbol(s)] for s in PRIORITY_STOCKS if norm_symbol(s) in sym_map][:4]
    names = [sym_map.get(norm_symbol(s), s) for s in names]
    names = [s for s in dict.fromkeys(names) if s in set(full["Symbol"])]
    sym_arr = full["Symbol"].values
    rec = []
    for sym in names:
        held = sym_arr == sym
        tr_l, va_l = tr_idx[~held[tr_idx]], va_idx[~held[va_idx]]
        te_l = TEST_IDX[held[TEST_IDX]]
        m = make_xgb(); m.fit(X_all[tr_l][:, sel_idx], y_all[tr_l].astype(int))
        thr_loso = pick_threshold(y_all[va_l].astype(int), m.predict_proba(X_all[va_l][:, sel_idx])[:, 1])
        p_te = m.predict_proba(X_all[te_l][:, sel_idx])[:, 1]
        mm = compute_metrics(y_all[te_l].astype(int), p_te, thr_loso)
        uni = per_stock.loc[per_stock["Stock"] == sym, "Accuracy"]
        rec.append({"Held-out stock": sym, "Test rows": len(te_l), "LOSO accuracy": mm["accuracy"], "LOSO ROC-AUC": mm["roc_auc"], "LOSO F1": mm["f1"],
                    "Universal model accuracy (stock seen in training)": float(uni.iloc[0]) if len(uni) else np.nan})
        print(f"Held out {sym:15s} | LOSO acc {mm['accuracy']:.4f} | AUC {fmt(mm['roc_auc'])} | (stock seen in training: {float(uni.iloc[0]) if len(uni) else float('nan'):.4f})")
    loso_df = pd.DataFrame(rec)
    display(loso_df.round(4))
    log_event("loso_run", stocks=names)
else:
    print("RUN_LOSO = False -> leave-one-stock-out experiment skipped (set True to run it).")

## 18. Prediction for `TARGET_STOCK` (latest available row)
This is a **live-style forecast for the trading day after the last date present in the dataset**. It is separate from the historical test evaluation above and its outcome is unknown (the dataset ends in 2024, so "latest" means the dataset's last row, not today).

In [ ]:
def lstm_predict_end(end_idx):
    seq = WindowSequence(X_l, np.zeros(len(full), dtype=np.float32), np.asarray(end_idx), WINDOW, LSTM_BATCH)
    return lstm_model.predict(seq, verbose=0).ravel()

lat_all = np.flatnonzero(SPLIT == "latest")
lat_t = lat_all[full["Symbol"].values[lat_all] == TARGET_SYMBOL]
assert len(lat_t) == 1, "target stock must have exactly one 'latest' row"
row = full.iloc[lat_t[0]]
p_up_x = float(ga_model.predict_proba(X_all[lat_t][:, sel_idx])[0, 1])
thr_x = MODELS["GA-XGBoost"]["thr"]
lstm_ok = pos_arr[lat_t[0]] >= WINDOW - 1
p_up_l = float(lstm_predict_end(lat_t)[0]) if lstm_ok else None
thr_l = MODELS["Universal LSTM"]["thr"]

def show_prediction(model_name, p_up, thr):
    print("-" * 56); print(f"{model_name}")
    print(f"Target Stock          : {TARGET_STOCK} (dataset symbol {TARGET_SYMBOL})")
    print(f"Latest Date           : {row['Date'].date()}")
    print(f"Latest Closing Price  : {row['Close']:.2f}")
    print(f"Probability UP        : {p_up:.4f}")
    print(f"Probability DOWN      : {1 - p_up:.4f}")
    print(f"Predicted Direction   : {'UP' if p_up >= thr else 'DOWN'}   (decision threshold {thr:.3f}, confidence {max(p_up, 1 - p_up):.1%})")
show_prediction("Universal GA-XGBoost", p_up_x, thr_x)
if p_up_l is not None:
    show_prediction("Universal LSTM", p_up_l, thr_l)
else:
    print("LSTM prediction unavailable: not enough history for a 60-day window.")
print("\nReminder: next-day direction is noisy; probabilities close to 0.5 mean the model has little edge on that day.")

# latest forecast for every stock in the universe (handy for a dashboard / the future Streamlit app)
lat_x = ga_model.predict_proba(X_all[lat_all][:, sel_idx])[:, 1]
lat_ok = pos_arr[lat_all] >= WINDOW - 1
lat_l = np.full(len(lat_all), np.nan)
if lat_ok.any():
    lat_l[lat_ok] = lstm_predict_end(lat_all[lat_ok])
latest_all = pd.DataFrame({"Symbol": full["Symbol"].values[lat_all], "Date": full["Date"].values[lat_all], "Close": full["Close"].values[lat_all],
                           "xgb_prob_up": lat_x, "xgb_direction": np.where(lat_x >= thr_x, "UP", "DOWN"),
                           "lstm_prob_up": lat_l, "lstm_direction": np.where(np.isnan(lat_l), "n/a", np.where(lat_l >= thr_l, "UP", "DOWN"))})
log_event("latest_predictions_done")

## 19. DATA LEAKAGE AUDIT
Two kinds of evidence are used: (a) **numerical tests** that recompute things independently (truncation test for features, independent target re-computation, sequence-boundary checks, scaler statistics), and (b) **procedural logs** recorded while the notebook ran (which split labels each stage actually touched, and the order in which the test set was opened). `OVERALL: ALL CHECKS PASSED` is printed only if every check really passes.

In [ ]:
audit = []
def check(name, passed, detail=""):
    audit.append({"check": name, "passed": bool(passed), "detail": detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {name}" + (f"   ({detail})" if detail else ""))

print("=" * 78); print("DATA LEAKAGE AUDIT"); print("=" * 78)
ev_first = {}
for e_ in EVENTS:
    ev_first.setdefault(e_["event"], e_["order"])
first_test_open = ev_first.get("test_opened", 10 ** 9)
selection_events = ["baseline_models_trained", "ga_done", "xgb_final_fit", "lstm_fit", "models_frozen"] + \
                   [e for e in ("tuning_done", "tuning_skipped") if e in ev_first]
order_ok = all(ev_first.get(e, 10 ** 9) < first_test_open for e in selection_events)

# 1) no negative shifts / no future references in the feature code
neg_patterns = [r"shift\(\s*-", r"periods\s*=\s*-", r"\.diff\(\s*-", r"pct_change\(\s*-", r"center\s*=\s*True"]
hits = [p_ for p_ in neg_patterns if re.search(p_, FEATURE_CODE)]
bad_names = [f for f in FEATURES if "future" in f.lower() or "target" in f.lower()]
check("No negative shifts in feature creation", not hits and not bad_names,
      f"scanned feature code for {len(neg_patterns)} look-ahead patterns; feature names containing future/target: {len(bad_names)}")

# 2) target is exactly Close[t+1] relative to Close[t]
ok_t = True
for sym, g in df_u.groupby("Symbol", sort=False):
    c_ = g["Close"].values
    ref = np.append((c_[1:] > c_[:-1]).astype(float), np.nan)
    ref_ret = np.append(c_[1:] / c_[:-1] - 1.0, np.nan)
    ok_t &= bool(np.allclose(ref, g["target"].values, equal_nan=True) and
                 np.allclose(ref_ret, g["future_return"].values, equal_nan=True, atol=1e-9) and np.isnan(g["target"].values[-1]))
ok_t &= bool(full.loc[full["split"].isin(["train", "val", "test"]), "target"].notna().all())
check("Target uses t+1 only", ok_t, "independently recomputed from Close for every stock; last row of each stock is unlabeled and excluded")

# 3) features use only historical information (truncation test)
test_syms = [TARGET_SYMBOL] + [s for s in UNIVERSE if s != TARGET_SYMBOL][:2]
trunc_ok, n_trunc = True, 0
for sym in test_syms:
    d_ = df_u[df_u["Symbol"] == sym]
    ft_full = build_features_for_stock(d_)
    for frac in (0.5, 0.7, 0.9):
        cut = int(len(d_) * frac)
        ft_cut = build_features_for_stock(d_.iloc[:cut])
        trunc_ok &= bool(np.allclose(ft_full.iloc[:cut].values, ft_cut.values, equal_nan=True, rtol=1e-5, atol=1e-6))
        n_trunc += 1
check("Features use only historical information", trunc_ok,
      f"{n_trunc} truncation tests: features for rows <= t are identical when all data after t is removed")

# 4) scaler fitted on training data only
sc_ok = (int(lstm_scaler.n_samples_seen_) == len(tr_idx) and
         np.allclose(lstm_scaler.mean_, X_l_raw[tr_idx].mean(axis=0), rtol=1e-4, atol=1e-5) and USED.get("scaler_fit") == {"train"})
check("Scaler fitted on training data only", sc_ok, f"fit on {int(lstm_scaler.n_samples_seen_):,} rows = number of training rows; split labels used: {sorted(USED.get('scaler_fit', []))}")

# 5) GA never saw test data
ga_ok = (USED.get("ga_fit") == {"train"} and USED.get("ga_fitness_val") == {"val"} and
         ev_first.get("ga_done", 10 ** 9) < first_test_open and
         not (set(ga_tr_idx) | set(ga_va_idx)) & set(TEST_IDX.tolist()))
check("GA never saw test data", ga_ok, f"GA train labels: {sorted(USED.get('ga_fit', []))}, GA fitness labels: {sorted(USED.get('ga_fitness_val', []))}")

# 6) XGBoost fitting / tuning never saw test data
xgb_ok = (USED.get("xgb_fit") == {"train"} and USED.get("threshold") == {"val"} and
          (not RUN_TUNING or (USED.get("tuning_fit") == {"train"} and USED.get("tuning_eval") == {"val"})) and
          ("tuning_done" in ev_first or "tuning_skipped" in ev_first) and order_ok)
check("XGBoost tuning never saw test data", xgb_ok, f"fit: {sorted(USED.get('xgb_fit', []))}, tuning: {sorted(USED.get('tuning_eval', [])) or 'skipped'}, thresholds: {sorted(USED.get('threshold', []))}")

# 7) LSTM validation did not use test data
lstm_ok_a = USED.get("lstm_train") == {"train"} and USED.get("lstm_val") == {"val"}
check("LSTM validation did not use test data", lstm_ok_a, f"LSTM train windows end in {sorted(USED.get('lstm_train', []))}, validation windows end in {sorted(USED.get('lstm_val', []))}")

# 8) test opened only after model selection
n_open_before_final = sum(1 for e_ in EVENTS if e_["event"] == "test_opened" and e_["order"] < ev_first.get("models_frozen", -1))
check("Test set was opened only after model selection", order_ok and n_open_before_final == 0 and
      EVENTS[first_test_open]["reason"].startswith("final evaluation"),
      "event order: " + " -> ".join(e_["event"] for e_ in EVENTS[:first_test_open + 1]))

# 9) validation data not used in the final test evaluation
check("Validation data did not enter final test evaluation", USED.get("final_eval") == {"test"} and not (set(TEST_IDX.tolist()) & set(va_idx.tolist())),
      f"final evaluation rows carry split labels {sorted(USED.get('final_eval', []))}")

# 10) no cross-stock sequences
ends = np.concatenate([lstm_tr_end, lstm_va_end, MODELS["Universal LSTM"]["test_eval_idx"]])
first_rows = ends - (WINDOW - 1)
cross_ok = bool((sym_codes[ends] == sym_codes[first_rows]).all() and (pos_arr[ends] - pos_arr[first_rows] == WINDOW - 1).all()
                and (date_int[ends] > date_int[first_rows]).all())
samp = np.random.default_rng(0).choice(ends, size=min(5000, len(ends)), replace=False)
cross_ok &= all(len(np.unique(sym_codes[e_ - WINDOW + 1:e_ + 1])) == 1 for e_ in samp)
check("No cross-stock sequences", cross_ok, f"{len(ends):,} windows: first and last row share the same stock; 5000 sampled windows checked row by row")

# 11) splits are chronological and per-stock
chron_ok = True
for _, r_ in split_table.iterrows():
    chron_ok &= (r_["train_end"] < r_["val_start"]) and (r_["val_end"] < r_["test_start"]) and (r_["train_start"] <= r_["train_end"])
ratio = split_table[["train_rows", "val_rows", "test_rows"]].div(split_table[["train_rows", "val_rows", "test_rows"]].sum(axis=1), axis=0)
chron_ok &= bool((ratio["train_rows"].between(0.65, 0.72)).all() and (ratio["test_rows"].between(0.12, 0.17)).all())
check("Splits are chronological", chron_ok, f"{len(split_table)} stocks: train < validation < test in time; mean split ratios "
      f"{ratio['train_rows'].mean():.2f}/{ratio['val_rows'].mean():.2f}/{ratio['test_rows'].mean():.2f}")

# 12) threshold fixed before the test set was opened
thr_ok = USED.get("threshold") == {"val"} and ev_first.get("models_frozen", 10 ** 9) < first_test_open and \
         EVENTS[ev_first["models_frozen"]].get("thresholds") is not None
check("Decision thresholds were fixed on validation data before test evaluation", thr_ok,
      "thresholds: " + ", ".join(f"{k}={v['thr']:.3f}" for k, v in MODELS.items()))

ALL_PASSED = all(a["passed"] for a in audit)
print("-" * 78)
print("OVERALL: ALL CHECKS PASSED" if ALL_PASSED else "OVERALL: SOME CHECKS FAILED - do not trust the reported results until this is fixed")

## 20. Save artifacts for the future Streamlit application
Saved into `/kaggle/working/universal_stock_project/` (see the final cell for which files to download). The artifact round-trip test at the end reloads the saved XGBoost model, LSTM and scaler and verifies they reproduce the in-memory predictions.

In [ ]:
def jdefault(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

def dump_json(obj, name):
    (PROJECT_DIR / name).write_text(json.dumps(obj, indent=2, default=jdefault))

# 1-2) models
ga_model.save_model(str(PROJECT_DIR / "universal_ga_xgboost.json"))
lstm_model.save(str(PROJECT_DIR / "universal_lstm.keras"))
if "GA-XGBoost (tuned)" in MODELS:
    MODELS["GA-XGBoost (tuned)"]["model"].save_model(str(PROJECT_DIR / "universal_ga_xgboost_tuned.json"))
# 3) scaler
joblib.dump(lstm_scaler, PROJECT_DIR / "universal_lstm_scaler.pkl")

# 4) exact feature order for inference
warm = (df_u.groupby("Symbol").size() - full.groupby("Symbol").size()).dropna()
dump_json({
    "all_features": FEATURES, "ga_selected_features": SELECTED, "lstm_features": LSTM_FEATURES, "ohlcv_features": OHLCV_FEATURES,
    "lstm_window": WINDOW, "lstm_clip_z": CLIP_Z,
    "notes": "Compute all_features with feature_engineering.compute_features / build_features_for_stock, then select ga_selected_features in this exact order."
}, "universal_features.json")

# 5) supported stocks
st_ = stats.loc[stats.index.isin(UNIVERSE)]
dump_json({
    "trained_on": [{"symbol": s, "ticker": norm_symbol(s), "rows": int(st_.loc[s, "n_rows"]),
                    "first_date": str(st_.loc[s, "first_date"].date()), "last_date": str(st_.loc[s, "last_date"].date())} for s in UNIVERSE],
    "other_dataset_symbols_not_in_training_universe": [s for s in all_symbols if s not in set(UNIVERSE)],
    "note": "The model is built from relative features only, so it can be applied to any stock with enough price history; "
            "reliability is only measured for the stocks in 'trained_on'."
}, "supported_stocks.json")

# 6) configuration
dump_json({
    "project_title": "Universal Stock Direction Prediction Using Genetic Algorithm and XGBoost with LSTM",
    "dataset_handle": DATASET_HANDLE, "data_file": DATA_FILE.name, "symbol_column": "Symbol",
    "target_stock_in_this_run": TARGET_SYMBOL, "quick_mode": QUICK_MODE, "seed": SEED, "use_adj_close": USE_ADJ_CLOSE,
    "split": {"train": TRAIN_FRAC, "val": VAL_FRAC, "test": round(1 - TRAIN_FRAC - VAL_FRAC, 4), "per_stock": True, "embargo_rows": EMBARGO_ROWS},
    "target_definition": "UP=1 if Close[t+1]/Close[t]-1 > 0 else DOWN=0",
    "thresholds": {k: v["thr"] for k, v in MODELS.items() if "thr" in v}, "threshold_mode": THRESHOLD_MODE,
    "xgb_params": XGB_PARAMS, "tuned_params": MODELS.get("GA-XGBoost (tuned)", {}).get("tuned_params"),
    "ga": {"population": GA_POP, "generations": GA_GENS, "crossover": GA_CROSSOVER, "mutation": GA_MUTATION, "elitism": GA_ELITISM,
           "tournament_k": GA_TOURNAMENT_K, "fitness": "validation classification accuracy (no feature penalty)",
           "ga_xgb_trees": GA_TREES, "ga_train_rows": len(ga_tr_idx), "ga_val_rows": len(ga_va_idx), "best_validation_accuracy": ga_best_fit},
    "lstm": {"window": WINDOW, "clip_z": CLIP_Z, "batch_size": LSTM_BATCH, "epochs_trained": len(lstm_hist["loss"]),
             "architecture": "LSTM64(seq)-Dropout0.2-LSTM32-Dropout0.2-Dense16relu-Dense1sigmoid"},
    "feature_engineering": {"module": "feature_engineering.py", "version": FEATURE_ENGINEERING_VERSION,
                            "talib_hilbert_features_used": any(f.startswith("ht_") for f in FEATURES),
                            "max_warmup_rows": int(warm.max()), "min_rows_needed_for_lstm_inference": int(warm.max()) + WINDOW,
                            "inference_recipe": ["sort one stock's OHLCV by Date", "feature_engineering.build_features_for_stock(df)",
                                                 "xgboost: take ga_selected_features of the LAST row -> predict_proba",
                                                 "lstm: take lstm_features of the last 60 rows -> scaler.transform -> clip to +/-clip_z -> model.predict"]},
    "n_features_before_ga": len(FEATURES), "n_features_after_ga": len(SELECTED),
    "universe_size": len(UNIVERSE), "leakage_audit_all_passed": ALL_PASSED,
    "package_versions": {"xgboost": xgb.__version__, "tensorflow": tf.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "disclaimer": "Educational project. Next-day direction is hard to predict; do not use for real trading decisions."
}, "universal_config.json")

# 7-11) tables
comp.to_csv(PROJECT_DIR / "universal_model_results.csv", index=False)
per_stock.to_csv(PROJECT_DIR / "per_stock_results.csv", index=False)
pred_df.to_csv(PROJECT_DIR / "test_predictions.csv", index=False)
fi.to_csv(PROJECT_DIR / "feature_importance.csv", index=False)
ga_history.to_csv(PROJECT_DIR / "ga_history.csv", index=False)
ablation.to_csv(PROJECT_DIR / "ablation_study.csv", index=False)
split_table.to_csv(PROJECT_DIR / "split_dates.csv", index=False)
latest_all.to_csv(PROJECT_DIR / "latest_predictions_all_stocks.csv", index=False)
dump_json(audit, "leakage_audit.json")
if len(tuning_log): tuning_log.to_csv(PROJECT_DIR / "xgb_tuning_log.csv", index=False)
if len(loso_df): loso_df.to_csv(PROJECT_DIR / "loso_results.csv", index=False)
dump_json({"lstm_history": lstm_hist, "events": EVENTS}, "run_log.json")

# ---- artifact round-trip self-test ----
rt = []
m_rt = XGBClassifier(); m_rt.load_model(str(PROJECT_DIR / "universal_ga_xgboost.json"))
xs = X_all[va_idx[:2000]][:, sel_idx]
rt.append(("XGBoost JSON reload", np.allclose(m_rt.predict_proba(xs)[:, 1], ga_model.predict_proba(xs)[:, 1], atol=1e-6)))
sc_rt = joblib.load(PROJECT_DIR / "universal_lstm_scaler.pkl")
rt.append(("Scaler pickle reload", np.allclose(sc_rt.transform(X_l_raw[va_idx[:500]]), lstm_scaler.transform(X_l_raw[va_idx[:500]]))))
lm_rt = tf.keras.models.load_model(str(PROJECT_DIR / "universal_lstm.keras"))
eb = lstm_va_end[:256]
sq = WindowSequence(X_l, y_all, eb, WINDOW, 256)
rt.append(("LSTM .keras reload", np.allclose(lm_rt.predict(sq, verbose=0).ravel(), lstm_model.predict(sq, verbose=0).ravel(), atol=1e-4)))
for n_, ok_ in rt:
    print(f"[{'PASS' if ok_ else 'FAIL'}] artifact round-trip: {n_}")

zip_path = shutil.make_archive(str(WORK_DIR / "universal_stock_project"), "zip", PROJECT_DIR)
print("\nFiles in", PROJECT_DIR)
for p in sorted(PROJECT_DIR.iterdir()):
    print(f"  {p.name:42s} {p.stat().st_size / 1024:10.1f} KB")
print("\nEverything zipped for one-click download:", zip_path)

## 21. Research comparison - what is (and is not) the same as the paper
| | **Reference paper** - *Prediction of stock price direction using a hybrid GA-XGBoost algorithm* | **This project (an adaptation)** |
|---|---|---|
| Market / data | KOSPI-based experiment | NSE (India) stocks, Kaggle `nse-india-stock-market-data-2015-2024` |
| Scope | Index-level / single series experiment | **Many stocks, one universal model** (per-stock chronological split, then concatenated) |
| Stage 1 - feature-set expansion | OHLCV expanded with a broad technical-indicator set | ~80 indicators built per stock in relative (scale-free) form |
| Stage 2 - data preparation | Cleaning / preparation | Duplicate removal, per-stock features, leakage-safe splits and scaling |
| Stage 3 - feature selection | Genetic Algorithm selects a subset | GA with population 8, 30 generations, crossover 0.50, mutation 0.375, elitism 1; fitness = validation accuracy |
| Classifier | XGBoost, next-day direction | XGBoost (`n_estimators=100`), next-day direction |
| Deep learning | - | **Universal LSTM** (60-day windows, no cross-stock sequences) as a comparison |
| Evaluation | As reported in the paper | Untouched per-stock chronological test sets; accuracy, precision, recall, F1, ROC-AUC, per-stock table |

**Important:** this notebook is a methodological *adaptation*, not a reproduction of the paper's KOSPI experiment. Differences in market, period, universe, indicator implementation, target definition, split protocol and GA details mean the paper's reported accuracy neither transfers nor serves as a benchmark for the numbers produced here. Any claim in the project report should quote the test numbers from this notebook only.

### Reading the results for your report
* **Headline = untouched test accuracy** of GA-XGBoost and of the LSTM (never validation or training accuracy).
* Compare every model with the **majority baseline** and with the share of UP days: for next-day direction, a few points above ~50 % is already a meaningful signal.
* The GA's "best validation accuracy" is optimistic by construction (it is the maximum over many candidate subsets on that data).
* Per-stock accuracy varies a lot because each test period contains only a few hundred days; look at the mean/median/std across stocks rather than at a single stock.
* Ablation: OHLCV → expanded indicators → GA subset → final GA-XGBoost → LSTM shows what each ingredient actually contributes (it may be little or even negative; report whatever the numbers show).

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================
gx, ls_ = MODELS["GA-XGBoost"]["test_metrics"], MODELS["Universal LSTM"]["test_metrics"]
print("=" * 50)
print("UNIVERSAL STOCK MODEL SUMMARY")
print("=" * 50)
print(f"\nDataset: {DATASET_HANDLE}")
print(f"Number of stocks (universal model): {full['Symbol'].nunique()}   (dataset has {len(all_symbols)})")
print(f"Number of total rows (modelling frame): {len(full):,}   (clean dataset rows: {len(df):,})")
print(f"Date range: {df['Date'].min().date()} -> {df['Date'].max().date()}")
print(f"\nFeature count before GA: {len(FEATURES)}")
print(f"Feature count after GA : {len(SELECTED)}")
print(f"\nGA best validation accuracy: {ga_best_fit:.4f}   (optimistic by construction)")
print(f"\nGA-XGBoost test (untouched, {len(TEST_IDX):,} rows):")
print(f"  Accuracy : {gx['accuracy']:.4f}\n  Precision: {gx['precision']:.4f}\n  Recall   : {gx['recall']:.4f}\n  F1       : {gx['f1']:.4f}\n  ROC-AUC  : {fmt(gx['roc_auc'])}")
print(f"\nLSTM test (untouched, {len(MODELS['Universal LSTM']['test_eval_idx']):,} rows):")
print(f"  Accuracy : {ls_['accuracy']:.4f}\n  Precision: {ls_['precision']:.4f}\n  Recall   : {ls_['recall']:.4f}\n  F1       : {ls_['f1']:.4f}\n  ROC-AUC  : {fmt(ls_['roc_auc'])}")
print(f"\nMajority-class baseline test accuracy: {MODELS['Majority Baseline']['test_metrics']['accuracy']:.4f}")
print(f"\nAverage per-stock accuracy   : {stock_stats['mean']:.4f}")
print(f"Median per-stock accuracy    : {stock_stats['median']:.4f}")
print(f"Standard deviation           : {stock_stats['std']:.4f}   (min {stock_stats['min']:.4f}, max {stock_stats['max']:.4f})")
print(f"\nTarget stock: {TARGET_STOCK} ({TARGET_SYMBOL})   latest date {row['Date'].date()}")
print(f"Probability UP  : {p_up_x:.4f}  (GA-XGBoost)")
print(f"Probability DOWN: {1 - p_up_x:.4f}")
print(f"Predicted direction: {'UP' if p_up_x >= thr_x else 'DOWN'}")
if p_up_l is not None:
    print(f"LSTM: P(UP) = {p_up_l:.4f} -> {'UP' if p_up_l >= thr_l else 'DOWN'}")
print(f"\nLeakage audit: {'ALL CHECKS PASSED' if ALL_PASSED else 'SOME CHECKS FAILED'}")
if gx["accuracy"] > 0.70 or ls_["accuracy"] > 0.70:
    print("\n!! Test accuracy above 70 % for next-day direction is extraordinary. Re-inspect the leakage audit and the data before trusting it.")

print("\n" + "=" * 70)
print("FILES TO DOWNLOAD FOR THE STREAMLIT APP (from /kaggle/working/universal_stock_project/)")
print("=" * 70)
for fname, why in [("universal_ga_xgboost.json", "GA-XGBoost model"), ("universal_lstm.keras", "LSTM model"),
                   ("universal_lstm_scaler.pkl", "LSTM StandardScaler"), ("universal_features.json", "exact feature order (all / GA-selected / LSTM)"),
                   ("universal_config.json", "thresholds, window, clip, inference recipe"), ("feature_engineering.py", "identical feature code for inference"),
                   ("supported_stocks.json", "stock list for the dropdown"), ("feature_importance.csv", "feature-importance panel")]:
    print(f"  REQUIRED  {fname:30s} {why}")
for fname, why in [("per_stock_results.csv", "per-stock reliability table"), ("universal_model_results.csv", "model comparison table"),
                   ("ablation_study.csv", "ablation table"), ("latest_predictions_all_stocks.csv", "precomputed latest forecasts"),
                   ("ga_history.csv", "GA convergence data"), ("test_predictions.csv", "all test predictions"), ("*.png", "report figures")]:
    print(f"  optional  {fname:30s} {why}")
print("\nOr simply download universal_stock_project.zip (everything above in one file).")
print("The Streamlit app also needs OHLCV price history for the chosen stock (e.g. the same Kaggle CSV or a live data source).")